# Phase 2b: Preprocessing the Real Clinical Dataset

This notebook continues directly from `Phase2_Real_Data_EDA.ipynb`. Based on what the EDA uncovered, we make and justify four preprocessing decisions before any modeling:

1. **BMI missingness** is not random (19.9% stroke rate when missing vs 4.26% when present) — we preserve this signal with a missingness indicator, then impute.
2. **`gender = "Other"`** appears in only 1 of 5,110 rows — too sparse to encode as its own category.
3. **Categorical encoding** for `gender`, `ever_married`, `work_type`, `Residence_type`, `smoking_status`.
4. **Train/test split** done *before* imputation, to avoid leaking test-set statistics into training (a common real-data pitfall absent from the synthetic dataset, which needed no such care).

## 1. Load Data

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

pd.set_option('display.max_columns', None)

df = pd.read_csv("healthcare-dataset-stroke-data.csv")
df = df.drop(columns=["id"])  # identifier only, not a predictive feature
print("Shape after dropping id:", df.shape)
display(df.head())

Shape after dropping id: (5110, 11)


,gender,age,hypertension,heart_disease,ever_married,work_type,Residence_type,avg_glucose_level,bmi,smoking_status,stroke
0,Male,67.0,0,1,Yes,Private,Urban,228.69,36.6,formerly smoked,1
1,Female,61.0,0,0,Yes,Self-employed,Rural,202.21,NaN,never smoked,1
2,Male,80.0,0,1,Yes,Private,Rural,105.92,32.5,never smoked,1
3,Female,49.0,0,0,Yes,Private,Urban,171.23,34.4,smokes,1
4,Female,79.0,1,0,Yes,Self-employed,Rural,174.12,24.0,never smoked,1


## 2. Handle the Rare `gender = "Other"` Category

Only 1 row out of 5,110 has `gender = "Other"`. With so few examples, this category can't be learned reliably by any model, and one-hot encoding it would create a near-useless column. We drop this single row rather than recoding it into "Male" or "Female", since reassigning it would fabricate information we don't have.

In [2]:
print("Rows before dropping rare gender category:", len(df))
df = df[df["gender"] != "Other"].reset_index(drop=True)
print("Rows after:", len(df))
print(df["gender"].value_counts())

Rows before dropping rare gender category: 5110
Rows after: 5109
gender
Female    2994
Male      2115
Name: count, dtype: int64


## 3. Train/Test Split (Before Imputation)

We split *before* computing the BMI median for imputation. If we computed the median on the full dataset first, information from the test set would leak into training (the model would benefit from a statistic it shouldn't have access to at training time). This wasn't a concern in Phase 1's synthetic data, which had no missing values at all.

We use `stratify=y` to preserve the same severe class imbalance (≈4.9% positive) in both the train and test sets — without this, a random split could accidentally concentrate even fewer positive cases into the test set, making evaluation unreliable.

In [3]:
X = df.drop(columns=["stroke"])
y = df["stroke"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Train shape:", X_train.shape, " Positive rate:", y_train.mean().round(4))
print("Test shape:", X_test.shape, " Positive rate:", y_test.mean().round(4))

Train shape: (4087, 10)  Positive rate: 0.0487
Test shape: (1022, 10)  Positive rate: 0.0489


## 4. BMI Missingness: Indicator + Median Imputation

Step 1: create `bmi_missing` (1 if BMI was missing, 0 otherwise) **before** imputing, so the model can still use the fact that BMI was missing as a feature in its own right.

Step 2: impute missing BMI values using the **median computed on the training set only**, then apply that same training-set median to the test set. Median is used instead of mean because BMI is right-skewed (skew ≈ 1.06), so the median is less distorted by the small number of very high outliers (max BMI = 97.6).

In [4]:
# Step 1: missingness indicator (computed independently on each split, just flagging presence/absence)
X_train = X_train.copy()
X_test = X_test.copy()
X_train["bmi_missing"] = X_train["bmi"].isna().astype(int)
X_test["bmi_missing"] = X_test["bmi"].isna().astype(int)

print("Train bmi_missing rate:", X_train["bmi_missing"].mean().round(4))
print("Test bmi_missing rate:", X_test["bmi_missing"].mean().round(4))

# Step 2: median imputation using TRAIN median only (avoids test-set leakage)
train_bmi_median = X_train["bmi"].median()
print("\nTraining-set BMI median used for imputation:", train_bmi_median)

X_train["bmi"] = X_train["bmi"].fillna(train_bmi_median)
X_test["bmi"] = X_test["bmi"].fillna(train_bmi_median)

print("\nRemaining missing values (train):", X_train.isna().sum().sum())
print("Remaining missing values (test):", X_test.isna().sum().sum())

Train bmi_missing rate: 0.0399
Test bmi_missing rate: 0.0372

Training-set BMI median used for imputation: 28.1

Remaining missing values (train): 0
Remaining missing values (test): 0


## 5. Encoding Categorical Variables

- `gender`: binary (Male/Female after dropping "Other") → label-encode as 0/1.
- `ever_married`: binary (Yes/No) → label-encode as 0/1.
- `Residence_type`: binary (Urban/Rural) → label-encode as 0/1.
- `work_type` (5 categories) and `smoking_status` (4 categories, including the informative "Unknown") → one-hot encode, since they have no natural ordering.

We fit the encoders on the training set only, then apply the same mapping to the test set — again to avoid any leakage of test-set category frequencies into training.

In [5]:
binary_maps = {
    "gender": {"Male": 0, "Female": 1},
    "ever_married": {"No": 0, "Yes": 1},
    "Residence_type": {"Rural": 0, "Urban": 1},
}

for col, mapping in binary_maps.items():
    X_train[col] = X_train[col].map(mapping)
    X_test[col] = X_test[col].map(mapping)

onehot_cols = ["work_type", "smoking_status"]
X_train = pd.get_dummies(X_train, columns=onehot_cols, drop_first=False)
X_test = pd.get_dummies(X_test, columns=onehot_cols, drop_first=False)

# Align test columns to train columns (in case a rare category only appears in one split)
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

print("Final feature count:", X_train.shape[1])
print(X_train.columns.tolist())

Final feature count: 18
['gender', 'age', 'hypertension', 'heart_disease', 'ever_married', 'Residence_type', 'avg_glucose_level', 'bmi', 'bmi_missing', 'work_type_Govt_job', 'work_type_Never_worked', 'work_type_Private', 'work_type_Self-employed', 'work_type_children', 'smoking_status_Unknown', 'smoking_status_formerly smoked', 'smoking_status_never smoked', 'smoking_status_smokes']


In [6]:
display(X_train.head())

,gender,age,hypertension,heart_disease,ever_married,Residence_type,avg_glucose_level,bmi,bmi_missing,work_type_Govt_job,work_type_Never_worked,work_type_Private,work_type_Self-employed,work_type_children,smoking_status_Unknown,smoking_status_formerly smoked,smoking_status_never smoked,smoking_status_smokes
845,1,48.0,0,0,1,1,69.21,33.1,0,False,False,True,False,False,False,False,True,False
3744,1,29.0,0,0,0,1,84.19,21.2,0,False,False,True,False,False,False,False,True,False
4183,1,35.0,0,0,1,0,119.40,22.9,0,False,False,True,False,False,False,False,True,False
3409,0,38.0,0,0,1,0,108.68,32.7,0,False,False,True,False,False,False,False,True,False
284,0,14.0,0,0,0,1,82.34,31.6,0,True,False,False,False,False,True,False,False,False


## 6. Final Sanity Checks

In [7]:
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("Any missing values left in X_train?", X_train.isna().sum().sum())
print("Any missing values left in X_test?", X_test.isna().sum().sum())
print("Train columns == Test columns?", list(X_train.columns) == list(X_test.columns))
print()
print("y_train positive rate:", y_train.mean().round(4))
print("y_test positive rate:", y_test.mean().round(4))

X_train shape: (4087, 18)
X_test shape: (1022, 18)
Any missing values left in X_train? 0
Any missing values left in X_test? 0
Train columns == Test columns? True

y_train positive rate: 0.0487
y_test positive rate: 0.0489


In [8]:
# Save the processed splits for the modeling notebook (Phase 2c)
import joblib

joblib.dump({
    "X_train": X_train, "X_test": X_test,
    "y_train": y_train, "y_test": y_test,
    "train_bmi_median": train_bmi_median,
    "feature_names": list(X_train.columns),
}, "phase2_preprocessed_data.joblib")

print("Saved preprocessed train/test splits to phase2_preprocessed_data.joblib")

Saved preprocessed train/test splits to phase2_preprocessed_data.joblib


## 7. Preprocessing Summary

| Decision | Action | Why |
|---|---|---|
| `id` column | Dropped | Pure identifier, no predictive value |
| `gender = "Other"` (1 row) | Row dropped | Too sparse to encode or learn from reliably |
| BMI missing (201 rows) | Indicator (`bmi_missing`) + median imputation using **train-set median only** | Missingness itself correlates with stroke (19.9% vs 4.26%); dropping rows would discard that signal, and computing the median on the full dataset would leak test information into training |
| `gender`, `ever_married`, `Residence_type` | Binary label encoding | Naturally binary, no ordinal meaning needed |
| `work_type`, `smoking_status` | One-hot encoding | Multiple unordered categories; "Unknown" smoking status kept as its own category since it's informative, not random |
| Train/test split | Stratified 80/20, split **before** imputation | Preserves the rare positive class proportion in both sets; prevents leakage of test statistics into preprocessing |

The data is now ready for classification modeling (Phase 2c), where we will train models with the class imbalance explicitly accounted for, and evaluate using Recall and ROC-AUC rather than raw accuracy.